# تدريب نماذج التعلم العميق — ذكاء سوق العمل اليمني
**Runtime → Change runtime type → GPU (T4)** قبل البدء. يعمل على CPU أيضاً لكن ببطء شديد لـ AraBERT.

الخطوات: (1) الكود (2) البيانات (3) تدقيق (4) خطوط أساس (5) AraBERT تصنيف ×2 (6) AraBERT NER (7) LSTM (8) ملخص (9) تنزيل النتائج إلى الكمبيوتر.
- لا يُستخدم Google Drive: الملفات تُرفع من جهازك بـ `files.upload` وتُنزّل إليه بـ `files.download`.
- بيانات test لا تُستخدم في التدريب أو اختيار النموذج — كل سكربت تدريب يقرأ train/validation فقط.

In [ ]:
import os, sys, subprocess, json, zipfile, io, shutil
ROOT = "/content/lmi"            # ← نفس مجلد مشروعك إن كان موجوداً من خلية run_all
os.makedirs(ROOT, exist_ok=True); os.chdir(ROOT)
try:
    import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "لا يوجد — CPU")
except ImportError: pass
!pip -q install "transformers>=4.44" seqeval sklearn-crfsuite arabert accelerate safetensors

## 1) الكود
إن كان مجلد `src/` موجوداً (نفس الجلسة) تُتخطى هذه الخطوة. وإلا ارفع **`lmi_dl_code.zip`** من مجلد Downloads.

In [ ]:
from google.colab import files
if not os.path.exists(f"{ROOT}/src/training/train_classifier.py"):
    up = files.upload()                                   # اختر lmi_dl_code.zip
    name = next(iter(up)); zipfile.ZipFile(io.BytesIO(up[name])).extractall(ROOT)   # من الذاكرة (يتجنب «lmi (1).zip»)
print("src:", os.path.exists(f"{ROOT}/src/training/train_classifier.py"))

## 2) البيانات النهائية
إن كان `FINAL_TRAINING_DATASET` موجوداً تُتخطى. وإلا ارفع **`lmi_train_data.zip`** من Downloads (≈41MB: ملفات التصنيف وNER وGold + تقرير التدقيق).

In [ ]:
D = f"{ROOT}/FINAL_TRAINING_DATASET"
NEED = ["classification_opportunity_type/train.csv", "classification_geographic_scope/train.csv", "ner/train.jsonl"]
if not all(os.path.exists(f"{D}/{x}") for x in NEED):
    up = files.upload()                                   # اختر lmi_train_data.zip
    name = next(iter(up)); zipfile.ZipFile(io.BytesIO(up[name])).extractall(ROOT)
assert all(os.path.exists(f"{D}/{x}") for x in NEED), "البيانات غير موجودة — لا يتم التدريب على بيانات بديلة"
print("files:", len(os.listdir(D)))

## 3) تدقيق البيانات (جودة · تسرب · NER · Gold · كفاية LSTM)
إن ظهر **FAIL** (تسرب بين التقسيمات) أوقف وأصلح قبل التدريب.

In [ ]:
if os.path.exists(f"{D}/unique_opportunities.csv"):
    !python -m src.data.audit
else:
    print("التدقيق الكامل نُفّذ مسبقاً على البيانات الكاملة — قراءة التقرير المرفق")
rep = json.load(open(f"{ROOT}/reports/audit/audit_report.json", encoding="utf-8"))
FORCE = False
assert rep["verdict"] != "FAIL" or FORCE, "التدقيق فشل — راجع reports/audit/audit_report.md"
print("الحكم:", rep["verdict"], rep["counts"])

## 4) خطوط الأساس (TF-IDF + LR/SVM · CRF)
اختيار المعاملات على validation فقط، ثم test مرة واحدة. (اختياري إن كانت نُفّذت مسبقاً)

In [ ]:
RUN_BASELINES = False   # نُفّذت على CPU خارج Colab؛ اجعلها True لإعادتها
if RUN_BASELINES:
    !python -m src.training.baselines --task all

## 5) AraBERT — تصنيف نوع الفرصة والنطاق الجغرافي
أفضل Checkpoint بـ Macro-F1 على validation، ثم تقييم test مستقل.

In [ ]:
for task in ["opportunity_type", "geographic_scope"]:
    !python -m src.training.train_classifier --task {task}
    !python -m src.evaluation.evaluate_classifier --task {task}

## 6) AraBERT — استخراج الكيانات (NER، BIO)
محاذاة أول رمز فرعي، تقطيع النصوص الطويلة إلى مقاطع ≤180 كلمة، أفضل Checkpoint بـ Entity-F1 (seqeval صارم) على validation.

In [ ]:
!python -m src.training.train_ner
!python -m src.evaluation.evaluate_ner

## 7) LSTM — التنبؤ الزمني (مع فحص الكفاية ومقارنة Naive / MA-3)

In [ ]:
if os.path.exists(f"{D}/unique_opportunities.csv"):
    !python -m src.forecasting.train_lstm
else:
    print("LSTM دُرّب مسبقاً على CPU خارج Colab — تخطٍ")

## 8) ملخص النتائج (test)

In [ ]:
import pandas as pd
ev = f"{ROOT}/reports/evaluation"; rows = []
def J(p):
    p = f"{ev}/{p}/metrics.json"; return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else None
for t in ["opportunity_type", "geographic_scope"]:
    m, b = J(f"classification_{t}"), J(f"baseline_{t}")
    if m: rows.append({"task": t, "model": "AraBERT", "acc": m["accuracy"], "macro_f1": m["macro_f1"], "weighted_f1": m["weighted_f1"]})
    for k in ["tfidf_logreg", "tfidf_linearsvc"]:
        if b and k in b: rows.append({"task": t, "model": k, "acc": b[k]["test"]["accuracy"], "macro_f1": b[k]["test"]["macro_f1"], "weighted_f1": b[k]["test"]["weighted_f1"]})
n, bn = J("ner"), J("baseline_ner")
if n: rows.append({"task": "ner", "model": "AraBERT", "entity_f1": n["entity_f1"], "entity_p": n["entity_precision"], "entity_r": n["entity_recall"]})
if bn: rows.append({"task": "ner", "model": "CRF", "entity_f1": bn["crf"]["test"]["entity_f1"]})
display(pd.DataFrame(rows))
s = json.load(open(f"{ev}/lstm_summary.json", encoding="utf-8")) if os.path.exists(f"{ev}/lstm_summary.json") else {}
display(pd.DataFrame([{"family": f, "trained": v.get("trained"), "reason": v.get("reason"),
    **({k: v["test"][k]["mae"] for k in ["lstm","naive","moving_avg_3"]} if v.get("trained") else {})}
    for f, v in s.items() if isinstance(v, dict) and "trained" in v]))

## 9) نقل النتائج إلى الكمبيوتر (بدون Drive)
ملفان منفصلان: **lmi_reports.zip** (صغير — نزّله أولاً) ثم **lmi_models.zip** (الأوزان بنصف الدقة fp16 ≈ نصف الحجم).
يُحفظان في مجلد **Downloads** على جهازك؛ بعدها أخبر Claude «نزلت النتائج» ليكمل الربط بالتطبيق.
إن فشل تنزيل الملف الكبير: من لوحة الملفات يسار Colab → كليك يمين على `/content/lmi_models.zip` → Download.

In [ ]:
!python scripts/package_results.py --out /content --fp16
files.download("/content/lmi_reports.zip")

In [ ]:
files.download("/content/lmi_models.zip")